In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy faiss-cpu mteb

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os, sys, logging, json, time
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
sys.path.insert(0, "/content/drive/MyDrive/colbert_ru/code/")
sys.path.insert(0, "/content/drive/MyDrive/colbert_ru")

DRIVE = "/content/drive/MyDrive/colbert_ru"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

Device: cuda
GPU: Tesla T4


In [ ]:
DATA_SOURCE = "miracl_drive"
MAX_CORPUS_DOCS = None
MAX_QUERIES = None 

SKIP_COLBERT_WHEN_CORPUS_SUBSET = True


def _load_jsonl(path, max_lines=None):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            if max_lines is not None and i >= max_lines:
                break
            rows.append(json.loads(line))
    return rows


qrels: Dict[str, Dict[str, int]] = {}
latency_meta: Dict[str, Any] = {"data_source": DATA_SOURCE}

if DATA_SOURCE == "miracl_drive":
    corpus = _load_jsonl(f"{DRIVE}/datasets/corpus.jsonl", MAX_CORPUS_DOCS)
    corpus_texts = [d["text"] for d in corpus]
    corpus_ids = [d["id"] for d in corpus]
    queries_data = _load_jsonl(f"{DRIVE}/datasets/queries.jsonl", MAX_QUERIES)
    queries_list = [q["text"] for q in queries_data]
    with open(f"{DRIVE}/datasets/qrels.tsv") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 4:
                qid, _, did, rel = parts[:4]
                qrels.setdefault(qid, {})[did] = int(rel)
    latency_meta.update(
        max_corpus_docs=MAX_CORPUS_DOCS,
        max_queries=MAX_QUERIES,
    )


RUN_COLBERT = True
print(f"Corpus: {len(corpus_texts)} docs | Queries: {len(queries_list)} | RUN_COLBERT={RUN_COLBERT}")

Corpus: 35008 docs | Queries: 1252 | RUN_COLBERT=True


In [5]:
from evaluate import SystemMetrics

WARMUP = 50
TEST = 200
TOP_K = 100

all_benchmarks = {}

In [6]:
from index_and_retrieve import BM25Baseline

t0 = time.perf_counter()
bm25 = BM25Baseline(corpus_texts, corpus_ids)
bm25_index_time = time.perf_counter() - t0

sm_bm25 = SystemMetrics(f"{DRIVE}/index_finetuned")  # dummy dir, only for measure_latency
bm25_latency = sm_bm25.measure_latency(
    retrieve_fn=lambda q: bm25.retrieve(q, top_k=TOP_K),
    queries=queries_list,
    warmup=WARMUP,
    test=TEST,
)

all_benchmarks["BM25"] = {
    "index_build_time_sec": bm25_index_time,
    "index_size_mb": 0.0,  # BM25 in-memory, no persistent index
    **bm25_latency,
}
print("BM25:", json.dumps(all_benchmarks["BM25"], indent=2))

BM25: {
  "index_build_time_sec": 2.857229653000104,
  "index_size_mb": 0.0,
  "avg_latency_ms": 76.48188419498979,
  "p50_latency_ms": 70.92291999981626,
  "p95_latency_ms": 126.26293385005737,
  "p99_latency_ms": 168.84649962999876,
  "throughput_qps": 13.074991686273185
}


In [ ]:
from modeling import build_model_and_tokenizer
from index_and_retrieve import ColBERTRetriever
from config import ModelConfig

def benchmark_colbert(model_name, index_dir, checkpoint_path=None, is_colbert_xm=False):
    if is_colbert_xm:
        model_cfg = ModelConfig(encoder_name="facebook/xmod-base", embedding_dim=128)
    else:
        model_cfg = ModelConfig(encoder_name="xlm-roberta-base", embedding_dim=128)

    model, tokenizer = build_model_and_tokenizer(model_cfg)

    if is_colbert_xm:
        from huggingface_hub import hf_hub_download
        import safetensors.torch
        model_path = hf_hub_download("antoinelouis/colbert-xm", "model.safetensors")
        hf_state_dict = safetensors.torch.load_file(model_path)
        new_sd = {}
        for k, v in hf_state_dict.items():
            if k.startswith("roberta."):
                new_sd[k.replace("roberta.", "encoder.backbone.", 1)] = v
            elif k == "linear.weight":
                new_sd["encoder.linear.weight"] = v
            else:
                new_sd[k] = v
        model.load_state_dict(new_sd, strict=False)
        if hasattr(model.encoder.backbone, "set_default_language"):
            model.encoder.backbone.set_default_language("ru_RU")
    elif checkpoint_path:
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])

    retriever = ColBERTRetriever(model, tokenizer, index_dir, model_cfg, device=device)

    sm = SystemMetrics(index_dir)

    def encode_only(query):
        enc = tokenizer(
            query, max_length=model_cfg.query_max_len,
            padding="max_length", truncation=True, return_tensors="pt",
        )
        q_ids = enc["input_ids"].to(retriever.device)
        q_mask = enc["attention_mask"].to(retriever.device)
        with torch.no_grad():
            return model.encode_query(q_ids, q_mask)

    latency = sm.measure_latency(
        retrieve_fn=lambda q: retriever.retrieve(q, top_k=TOP_K),
        queries=queries_list,
        warmup=WARMUP,
        test=TEST,
        encode_fn=encode_only,
    )

    result = {
        "index_size_mb": sm.index_size_mb(),
        "avg_vectors_per_doc": sm.avg_vectors_per_doc(),
        **latency,
    }
    print(f"{model_name}: {json.dumps(result, indent=2)}")
    return result

In [9]:
if RUN_COLBERT:
    all_benchmarks["ColBERT-XM_zero-shot"] = benchmark_colbert(
        "ColBERT-XM_zero-shot",
        index_dir=f"{DRIVE}/index_zeroshot_colbert_xm_xmod",
        is_colbert_xm=True,
    )
else:
    print("Пропуск ColBERT-XM (см. RUN_COLBERT / MAX_CORPUS_DOCS).")

Loading weights:   0%|          | 0/4085 [00:00<?, ?it/s]

XmodModel LOAD REPORT from: facebook/xmod-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ColBERT-XM_zero-shot: {
  "index_size_mb": 1160.7281494140625,
  "avg_vectors_per_doc": 135.80730118829982,
  "avg_latency_ms": 3487.721115560015,
  "p50_latency_ms": 3294.0498519997163,
  "p95_latency_ms": 4184.359151449916,
  "p99_latency_ms": 4276.792396989913,
  "throughput_qps": 0.28672017253289833,
  "query_encode_latency_ms": 201.91858092999155
}


In [10]:
if RUN_COLBERT:
    all_benchmarks["ColBERT-RU_finetuned"] = benchmark_colbert(
        "ColBERT-RU_finetuned",
        index_dir=f"{DRIVE}/index_finetuned",
        checkpoint_path=f"{DRIVE}/checkpoints/checkpoint_phase2_best.pt",
    )
else:
    print("Пропуск ColBERT-RU (см. RUN_COLBERT / MAX_CORPUS_DOCS).")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ColBERT-RU_finetuned: {
  "index_size_mb": 1160.7281494140625,
  "avg_vectors_per_doc": 135.80730118829982,
  "avg_latency_ms": 3304.975716159991,
  "p50_latency_ms": 3131.359823000139,
  "p95_latency_ms": 3993.109468049988,
  "p99_latency_ms": 4082.249184119664,
  "throughput_qps": 0.3025740840122986,
  "query_encode_latency_ms": 9.665771560003122
}


In [11]:
from benchmark_dense_baselines import build_e5_baseline

def benchmark_dense(name, baseline_factory):
    """Build index + measure latency for a dense baseline."""
    baseline = baseline_factory(batch_size=64)

    build_time = baseline.build_index(
        corpus_texts, corpus_ids,
        index_dir=f"{DRIVE}/dense_index_{name.lower().replace('-', '_')}",
    )

    sm = SystemMetrics(f"{DRIVE}/index_finetuned")  # dummy, only for measure_latency
    latency = sm.measure_latency(
        retrieve_fn=baseline.retrieve_fn(top_k=TOP_K),
        queries=queries_list,
        warmup=WARMUP,
        test=TEST,
        encode_fn=baseline.encode_only_fn(),
    )

    result = {
        "index_build_time_sec": build_time,
        "index_size_mb": baseline.index_size_mb(),
        **latency,
    }
    print(f"{name}: {json.dumps(result, indent=2)}")
    return result

In [12]:
all_benchmarks["E5-base"] = benchmark_dense("E5-base", build_e5_baseline)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/547 [00:00<?, ?it/s]

E5-base: {
  "index_build_time_sec": 530.65130877,
  "index_size_mb": 102.5625,
  "avg_latency_ms": 22.943647210013296,
  "p50_latency_ms": 20.828451499710354,
  "p95_latency_ms": 38.950873999647214,
  "p99_latency_ms": 45.360408239712314,
  "throughput_qps": 43.58504952793948,
  "query_encode_latency_ms": 13.664559415024087
}


In [13]:
import pandas as pd

rows = []
for model_name, bm in all_benchmarks.items():
    rows.append({
        "Model": model_name,
        "Index Size (MB)": f"{bm.get('index_size_mb', 0):.1f}",
        "Build Time (s)": f"{bm.get('index_build_time_sec', 0):.1f}" if 'index_build_time_sec' in bm else "—",
        "Avg Latency (ms)": f"{bm['avg_latency_ms']:.1f}",
        "P50 (ms)": f"{bm.get('p50_latency_ms', 0):.1f}",
        "P95 (ms)": f"{bm['p95_latency_ms']:.1f}",
        "P99 (ms)": f"{bm.get('p99_latency_ms', 0):.1f}",
        "QPS": f"{bm['throughput_qps']:.1f}",
        "Encode Only (ms)": f"{bm.get('query_encode_latency_ms', 0):.1f}" if 'query_encode_latency_ms' in bm else "—",
    })

df = pd.DataFrame(rows).set_index("Model")
print("\n=== Latency & Efficiency Benchmark ===")
print(df.to_string())


=== Latency & Efficiency Benchmark ===
                     Index Size (MB) Build Time (s) Avg Latency (ms) P50 (ms) P95 (ms) P99 (ms)   QPS Encode Only (ms)
Model                                                                                                                 
BM25                             0.0            2.9             76.5     70.9    126.3    168.8  13.1                —
ColBERT-XM_zero-shot          1160.7              —           3487.7   3294.0   4184.4   4276.8   0.3            201.9
ColBERT-RU_finetuned          1160.7              —           3305.0   3131.4   3993.1   4082.2   0.3              9.7
E5-base                        102.6          530.7             22.9     20.8     39.0     45.4  43.6             13.7


In [15]:
print("\n=== Hardware Info ===")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
import platform
print(f"CPU: {platform.processor()}")
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
except ImportError:
    print("RAM: (install psutil for RAM info)")


=== Hardware Info ===
Device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB
CPU: x86_64
RAM: 12.7 GB
